<h1>Chapter 5 - Text Embeddings</h1>
<i>Generate embeddings for text and images that captures the meaning of it.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch05_text_embedding/text_embeddings.ipynb)

---

This notebook is for Chapter 5 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


In [2]:
!pip install sentence-transformers==4.1.0
!pip install requests==2.32.3
!pip install matplotlib==3.10.3
!pip install openai==1.79.0
!pip install langchain-openai==0.3.17
!pip install pandas==2.2.3
!pip install rank-bm25
!pip install langchain_text_splitters
!pip install openai==1.83.0
!pip install PyPDF2==3.0.1

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached scipy-1.17.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)
  Using cached pillow-12.1.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached filelock-3.24.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2026.2.19-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Usi

### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [6]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

Cloning into 'RAG-with-Python-Cookbook'...
remote: Enumerating objects: 499, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 499 (delta 29), reused 0 (delta 0), pack-reused 450 (from 2)
Receiving objects: 100% (499/499), 39.67 MiB | 17.60 MiB/s, done.
Resolving deltas: 100% (193/193), done.
/content/RAG-with-Python-Cookbook
Your branch is up to date with 'origin/main'.


### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

ModuleNotFoundError: No module named 'google'

### How to generate embeddings using the OpenAI and HuggingFace API

In [8]:
from openai import OpenAI

text_chunks = ["The sky is blue.", "The grass is green."]

client = OpenAI()  # uses the environment variable OPENAI_API_KEY

embeddings_list = []

for text_chunk in text_chunks:
    response = client.embeddings.create(
        input=[text_chunk], model="text-embedding-3-small"
    )
    embedding = response.data[0].embedding
    embeddings_list.append(embedding)

In [9]:
# tag::calculate_embeddings_using_openai_with_langchain[]
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
embeddings_langchain_list = []

for text_chunk in text_chunks:
    embeddings_openai_langchain = embeddings.embed_query(text_chunk)
    embeddings_langchain_list.append(embeddings_openai_langchain)

# end::calculate_embeddings_using_openai_with_langchain[]

In [ ]:
# from openai import OpenAI

# def calculate_embeddings_using_openai():
#     """
#     Returns:
#         embeddings (list): list of embeddings
#     """

#     # tag::calculate_embeddings_using_openai[]
#     from openai import OpenAI

#     text_chunks = ["The sky is blue.", "The grass is green."]

#     client = OpenAI()  # uses the environment variable OPENAI_API_KEY

#     embeddings_list = []

#     for text_chunk in text_chunks:
#         response = client.embeddings.create(
#             input=[text_chunk], model="text-embedding-3-small"
#         )
#         embedding = response.data[0].embedding
#         embeddings_list.append(embedding)

#     # end::calculate_embeddings_using_openai[]

#     # tag::calculate_embeddings_using_openai_with_langchain[]
#     from langchain_openai import OpenAIEmbeddings

#     embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
#     embeddings_langchain_list = []

#     for text_chunk in text_chunks:
#         embeddings_openai_langchain = embeddings.embed_query(text_chunk)
#         embeddings_langchain_list.append(embeddings_openai_langchain)

#     # end::calculate_embeddings_using_openai_with_langchain[]
#     return embeddings_list, embeddings_langchain_list


# # calculate embeddings using openai
# embeddings_list, embeddings_langchain_list = (
#     calculate_embeddings_using_openai()
# )

In [ ]:
print(embeddings_list)
print("Length of embeddings:", len(embeddings_list[0]))

print(embeddings_langchain_list)
print("Length of embeddings:", len(embeddings_langchain_list[0]))

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
text_chunks = ["The sky is blue.", "The grass is green."]

embeddings = model.encode(text_chunks)

In [ ]:

# def create_text_embeddings_using_huggingface():
#     '''
#     Returns:
#         embeddings (list): list of embeddings
#     '''
#     # tag::create_text_embeddings_using_huggingface[]
#     from sentence_transformers import SentenceTransformer

#     # Load the model
#     model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
#     text_chunks = ["The sky is blue.", "The grass is green."]

#     # Generate embeddings
#     embeddings = model.encode(text_chunks)

#     # end::create_text_embeddings_using_huggingface[]

#     return embeddings

# embeddings_huggingface = create_text_embeddings_using_huggingface()


In [ ]:
# print(embeddings_huggingface)
print("Length of embeddings:", len(embeddings_huggingface[0]))

In [ ]:
# tag::calculate_embeddings_using_openai[]
from openai import OpenAI

text_chunks = ["The sky is blue.", "The grass is green."]

client = OpenAI()  # uses the environment variable OPENAI_API_KEY

embeddings_list = []

for text_chunk in text_chunks:
    response = client.embeddings.create(
        input=[text_chunk], model="text-embedding-3-small"
    )
    embedding = response.data[0].embedding
    embeddings_list.append(embedding)

# end::calculate_embeddings_using_openai[]

In [ ]:
# tag::calculate_embeddings_using_openai_with_langchain[]
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
embeddings_langchain_list = []

for text_chunk in text_chunks:
    embeddings_openai_langchain = embeddings.embed_query(text_chunk)
    embeddings_langchain_list.append(embeddings_openai_langchain)

# end::calculate_embeddings_using_openai_with_langchain[]

In [ ]:

# def calculate_embeddings_using_openai():
#     """
#     Calculates text embeddings using both the OpenAI API and LangChain's OpenAIEmbeddings.

#     Returns:
#         embeddings_list (list): List of embeddings generated by the OpenAI API for each text chunk.
#         embeddings_langchain_list (list): List of embeddings generated by LangChain's OpenAIEmbeddings for each text chunk.
#     """
#     # tag::calculate_embeddings_using_openai[]
#     from openai import OpenAI

#     text_chunks = ["The sky is blue.", "The grass is green."]

#     client = OpenAI()  # uses the environment variable OPENAI_API_KEY

#     embeddings_list = []

#     for text_chunk in text_chunks:
#         response = client.embeddings.create(
#             input=[text_chunk], model="text-embedding-3-small"
#         )
#         embedding = response.data[0].embedding
#         embeddings_list.append(embedding)

#     # end::calculate_embeddings_using_openai[]

#     # tag::calculate_embeddings_using_openai_with_langchain[]
#     from langchain_openai import OpenAIEmbeddings

#     embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
#     embeddings_langchain_list = []

#     for text_chunk in text_chunks:
#         embeddings_openai_langchain = embeddings.embed_query(text_chunk)
#         embeddings_langchain_list.append(embeddings_openai_langchain)

#     # end::calculate_embeddings_using_openai_with_langchain[]
#     return embeddings_list, embeddings_langchain_list

# embeddings_list_openai, embeddings_langchain_list = calculate_embeddings_using_openai()

In [ ]:
# embeddings_langchain_list

### Calculating the Distance Between Embeddings

In [ ]:
import numpy as np
from numpy.linalg import norm
import pandas as pd
import os
from openai import OpenAI

text_chunks = [
    "The Great Fire of London in 1666 destroyed over 13,000 houses.",
    "Julius Caesar was assassinated on the Ides of March (March 15) in 44 BCE.",
    "The Black Death is estimated to have killed nearly one-third of the population.",
]

users_question = "Tell me something interesting about diseases in history"

embeddings_df = pd.DataFrame(text_chunks, columns=["text_chunk"])

client = OpenAI()
embeddings = []

def create_embeddings(text_chunk, client):
    embedding = (
        client.embeddings.create(input=[text_chunk], model="text-embedding-3-small")
        .data[0]
        .embedding
    )
    return embedding

# Apply function create_embeddings to the correct column
embeddings_df["embedding"] = embeddings_df["text_chunk"].apply(
    create_embeddings, client=client
)

users_question_embedding = create_embeddings(
    text_chunk=users_question, client=client
)

In [ ]:
# create a list to store the calculated cosine similarity
cos_sim = []

def calculate_cosine_similarity(text_chunk_embedding, users_question_embedding):
    A = text_chunk_embedding
    B = users_question_embedding

    # calculate the cosine similarity
    cosine = np.dot(A, B) / (norm(A) * norm(B))
    return cosine

# Apply function calculate_cosine_similarity to the correct column
embeddings_df["similarity"] = embeddings_df["embedding"].apply(
    calculate_cosine_similarity, users_question_embedding=users_question_embedding
)

In [ ]:
# def calculate_cosine_similarity():
#     """
#     Calculate the cosine distance between two embeddings using numpy
#     and the sentence-transformers library.
#     """
#     # tag::calculate_cosine_distance_embeddings[]
#     import numpy as np
#     from numpy.linalg import norm
#     import pandas as pd
#     import os
#     from openai import OpenAI

#     text_chunks = [
#         "The Great Fire of London in 1666 destroyed over 13,000 houses.",
#         "Julius Caesar was assassinated on the Ides of March (March 15) in 44 BCE.",
#         "The Black Death, is estimated to have killed nearly one-third of the \
#             population",
#     ]

#     users_question = "Tell me something interesting about diseases in history"

#     embeddings_df = pd.DataFrame(text_chunks, columns=["text_chunk"])

#     client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
#     embeddings = []

#     def create_embeddings(text_chunk, client):
#         embedding = (
#             client.embeddings.create(input=[text_chunk], model="text-embedding-3-small")
#             .data[0]
#             .embedding
#         )
#         return embedding

#     # Apply function create_embeddings to the correct column
#     embeddings_df["embedding"] = embeddings_df["text_chunk"].apply(
#         create_embeddings, client=client
#     )

#     users_question_embedding = create_embeddings(
#         text_chunk=users_question, client=client
#     )
#     # end::calculate_cosine_distance_embeddings[]

#     # tag::calculate_cosine_distance_numpy[]
#     # create a list to store the calculated cosine similarity
#     cos_sim = []

#     def calculate_cosine_similarity(text_chunk_embedding, users_question_embedding):
#         A = text_chunk_embedding
#         B = users_question_embedding

#         # calculate the cosine similarity
#         cosine = np.dot(A, B) / (norm(A) * norm(B))
#         return cosine

#     # Apply function calculate_cosine_distance to the correct column
#     embeddings_df["similarity"] = embeddings_df["embedding"].apply(
#         calculate_cosine_similarity, users_question_embedding=users_question_embedding
#     )
#     # end::calculate_cosine_distance_numpy[]

#     return embeddings_df, users_question

In [ ]:
# embeddings_df, users_question = calculate_cosine_similarity()

In [ ]:
# embeddings_df

In [ ]:
from sentence_transformers.util import cos_sim

document_embeddings=embeddings_df["embedding"].tolist()
query_embedding=embeddings_df["embedding"].tolist()

for document_embedding in document_embeddings:
    # Compute cosine_similarity between documents and query
    scores = cos_sim(document_embedding, query_embedding)

In [ ]:
# def calculate_cosine_distance_using_sentence_transformer(document_embeddings, query_embedding):
#     """
#     Calculate the cosine distance between document embeddings and a query embedding
#     using the sentence-transformers library.

#     Args:
#         document_embeddings (list): List of document embeddings.
#         query_embedding (list): Query embedding.

#     Returns:
#         scores (numpy.ndarray): Cosine similarity scores.
#     """
#     # tag::calculate_cosine_distance_using_sentence_transformer[]
#     from sentence_transformers.util import cos_sim

#     for document_embedding in document_embeddings:
#         # Compute cosine_similarity between documents and query
#         scores = cos_sim(document_embedding, query_embedding)
#     # end::calculate_cosine_distance_using_sentence_transformer[]
#     return scores

# scores = calculate_cosine_distance_using_sentence_transformer(
#     document_embeddings=embeddings_df["embedding"].tolist(),
#     query_embedding=embeddings_df["embedding"].tolist()
# )


In [ ]:
# scores

### How to reduce the dimensionality of embeddings to be able to plot them

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from openai import OpenAI

# Define text chunks
text_chunks = [
    "The sky is blue.",
    "The sun is shining.",
    "I love chocolate.",
    "Ice cream is delicious.",
    "Roses are red.",
    "Violets are blue.",
]

# Initialize OpenAI client
# Assumes the OPENAI_API_KEY environment variable is set
client = OpenAI()
embeddings = []

# Generate embeddings for text chunks
for text_chunk in text_chunks:
    response = client.embeddings.create(
        input=[text_chunk], model="text-embedding-3-small"
    )
    embeddings.append(response.data[0].embedding)

# Convert embeddings to a DataFrame
# Each row represents one text chunk, each column one embedding dimension
embeddings_df = pd.DataFrame(
    embeddings, columns=[f"dim_{i}" for i in range(len(embeddings[0]))]
)

In [ ]:
# Perform PCA with 2 components
pca = PCA(n_components=2)
df_reduced = pca.fit_transform(embeddings_df)

# Create a new DataFrame with reduced dimensions
df_reduced = pd.DataFrame(df_reduced, columns=["PC1", "PC2"])
df_reduced["text"] = text_chunks  # Add original text chunks for labeling

# Create a scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(df_reduced["PC1"], df_reduced["PC2"])

# Add labels to each point
for i, label in enumerate(df_reduced["text"]):
    plt.text(df_reduced["PC1"][i], df_reduced["PC2"][i], label, fontsize=9)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Scatter Plot")

# Save the plot
plt.savefig("../principal_component_plot.svg", format="svg")

In [ ]:
# import sys
# import pandas as pd
# import os

# def create_pca_plot():
#     """
#     Reduces the dimensionality of text embeddings using PCA and visualizes them in a 2D scatter plot.

#     Returns:
#         df_reduced (DataFrame): DataFrame containing the reduced 2D coordinates and corresponding text labels.
#     """
#     # tag::calculate_embeddings_pca[]
#     import os
#     import pandas as pd
#     import matplotlib.pyplot as plt
#     from sklearn.decomposition import PCA
#     from openai import OpenAI

#     # Define text chunks
#     text_chunks = [
#         "The sky is blue.",
#         "The sun is shining.",
#         "I love chocolate.",
#         "Ice cream is delicious.",
#         "Roses are red.",
#         "Violets are blue.",
#     ]

#     # Initialize OpenAI client
#     client = OpenAI()  # uses the environment variable OPENAI_API_KEY
#     embeddings = []

#     # Generate embeddings for text chunks
#     for text_chunk in text_chunks:
#         response = client.embeddings.create(
#             input=[text_chunk], model="text-embedding-3-small"
#         )
#         embedding = response.data[0].embedding
#         embeddings.append(embedding)

#     # Convert embeddings to a DataFrame
#     embeddings_df = pd.DataFrame(
#         embeddings, columns=[f"dim_{i}" for i in range(len(embeddings[0]))]
#     )
#     # end::calculate_embeddings_pca[]

#     # tag::calculate_pca[]
#     # Perform PCA with 2 components
#     pca = PCA(n_components=2)
#     df_reduced = pca.fit_transform(embeddings_df)

#     # Create a new DataFrame with reduced dimensions
#     df_reduced = pd.DataFrame(df_reduced, columns=["PC1", "PC2"])
#     df_reduced["text"] = text_chunks  # Add original text chunks for labeling

#     # Function to create a scatter plot
#     plt.figure(figsize=(8, 6))
#     plt.scatter(df_reduced["PC1"], df_reduced["PC2"])

#     # Add labels to each point
#     for i, label in enumerate(df_reduced["text"]):
#         plt.text(df_reduced["PC1"][i], df_reduced["PC2"][i], label, fontsize=9)

#     # Add labels and title
#     plt.xlabel("Principal Component 1")
#     plt.ylabel("Principal Component 2")
#     plt.title("PCA Scatter Plot")

#     # Save the plot
#     plt.savefig("../principal_component_plot.svg", format="svg")
#     # end::calculate_pca[]
#     return plt, df_reduced

# plt = create_pca_plot()


### How to calculate multimodal embeddings using CLIP

In [ ]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# Load the model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Text and image inputs
descriptions = ["A photo of a cat", "A photo of a dog"]
images = [
    Image.open("../datasets/images/cat.jpg"),
    Image.open("../datasets/images/dog.jpg"),
]

# Disable gradient tracking since this code only runs inference
with torch.no_grad():
    inputs = processor(
        text=descriptions,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )
    outputs = model(**inputs)

dot_products_per_text = outputs.logits_per_text

# Calculate probabilities
probabilities = dot_products_per_text.softmax(dim=1)

In [ ]:
# def create_embeddings_using_clip():
#     # tag::create_embeddings_using_clip[]
#     import torch
#     from PIL import Image
#     from transformers import CLIPProcessor, CLIPModel

#     # Load the model and processor
#     model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
#     processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

#     # Text and image inputs
#     descriptions = ["A photo of a cat", "A photo of a dog"]
#     images = [
#         Image.open("../datasets/images/cat.jpg"),
#         Image.open("../datasets/images/dog.jpg"),
#     ]

#     # Inference
#     with torch.no_grad():
#         inputs = processor(
#             text=descriptions,
#             images=images,
#             return_tensors="pt",
#             padding=True,
#             truncation=True,
#         )
#         outputs = model(**inputs)

#     dot_products_per_text = outputs.logits_per_text

#     # Calculate probabilities
#     probabilities = dot_products_per_text.softmax(dim=1)
#     # end::create_embeddings_using_clip[]

#     return probabilities, dot_products_per_text



In [ ]:
# probabilities, dot_products_per_text = create_embeddings_using_clip()

In [ ]:
print(probabilities)

print("Probability that the first image is a cat:", probabilities[0][0].item())
print("Probability that the second image is a dog:", probabilities[1][1].item())

tensor([[9.9995e-01, 5.3788e-05],
        [1.2071e-01, 8.7929e-01]])
Probability that the first image is a cat: 0.9999462366104126
Probability that the second image is a dog: 0.8792924284934998


### How to use embeddings to train text classification models

In [ ]:
from rank_bm25 import BM25Okapi
import pandas as pd

text_chunks = [
    "The Great Fire of London in 1666 destroyed over 13,000 houses.",
    "Julius Caesar was assassinated on the Ides of March (March 15) in 44 BCE.",
    "The Black Death is estimated to have killed nearly one-third of the European population.",
]

# Tokenize text into words
tokenized_chunks = [chunk.split(" ") for chunk in text_chunks]

bm25 = BM25Okapi(tokenized_chunks)

user_query = "Tell me something interesting about diseases in history"
tokenized_query = user_query.split(" ")

# BM25 scores for each document
bm25_scores = bm25.get_scores(tokenized_query)

# Document IDs ordered by keyword relevance (best first)
keyword_ranking_doc_ids = (
    pd.DataFrame(bm25_scores, columns=["score"])
    .sort_values(by="score", ascending=False)
    .index
    .to_list()
)


In [24]:
from sentence_transformers import SentenceTransformer

embeddings_df = pd.DataFrame(text_chunks, columns=["text_chunk"])

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
embeddings = []

def create_embeddings(text_chunk, client):
    embedding = (
        client.embeddings.create(input=[text_chunk], model="text-embedding-3-small")
        .data[0]
        .embedding
    )
    return embedding

embeddings_df["embedding"] = embeddings_df["text_chunk"].apply(
    create_embeddings, client=client
)

users_question_embedding = create_embeddings(text_chunk=user_query, client=client)

# Compute cosine_similarity between documents and query
embeddings_df["similarity"] = cos_sim(
    embeddings_df["embedding"], users_question_embedding
)

ranking_semantic_search = embeddings_df.sort_values(
    by=["similarity"], ascending=False
).index


In [ ]:
# import sys
# import pandas as pd
# import os

# def hybrid_search():
#     """
#     This function is used in chapter 3: Embeddings for the
#     recipe hybrid search. It is combining keyword search using the
#     BM25 algorithm and semantic search using embedding models and
#     cosine similarity.
#     """
#     # bm25 keyword search algorithm
#     # tag::hybrid_search_bm25[]
#     from rank_bm25 import BM25Okapi

#     text_chunks = [
#         "The Great Fire of London in 1666 destroyed over 13,000 houses.",
#         "Julius Caesar was assassinated on the Ides of March (March 15) in 44 BCE.",
#         """The Black Death, is estimated to have killed nearly one-third of the European
#         population""",
#     ]
#     tokenized_chunks = [chunk.split(" ") for chunk in text_chunks]

#     bm25 = BM25Okapi(tokenized_chunks)

#     users_query = "Tell me something interesting about diseases in history"
#     tokenized_query = users_query.split(" ")

#     doc_scores = bm25.get_scores(tokenized_query)

#     ranking_keyword_search = (
#         pd.DataFrame(doc_scores, columns=["scores"])
#         .sort_values(by="scores", ascending=False)
#         .index.to_list()
#     )
#     # end::hybrid_search_bm25[]

#     # tag::hybrid_search_semantic_search[]
#     from sentence_transformers.util import cos_sim

#     embeddings_df = pd.DataFrame(text_chunks, columns=["text_chunk"])

#     client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
#     embeddings = []

#     def create_embeddings(text_chunk, client):
#         embedding = (
#             client.embeddings.create(input=[text_chunk], model="text-embedding-3-small")
#             .data[0]
#             .embedding
#         )
#         return embedding

#     embeddings_df["embedding"] = embeddings_df["text_chunk"].apply(
#         create_embeddings, client=client
#     )

#     users_question_embedding = create_embeddings(text_chunk=users_query, client=client)

#     # Compute cosine_similarity between documents and query
#     embeddings_df["similarity"] = cos_sim(
#         embeddings_df["embedding"], users_question_embedding
#     )

#     ranking_semantic_search = embeddings_df.sort_values(
#         by=["similarity"], ascending=False
#     ).index
#     # end::hybrid_search_semantic_search[]

#     return ranking_semantic_search, ranking_keyword_search


In [ ]:
# ranking_semantic_search, ranking_keyword_search = hybrid_search()

In [ ]:
ranking_semantic_search, ranking_keyword_search

(Index([2, 0, 1], dtype='int64'), [0, 1, 2])

In [22]:
#  calculate a combined similarity score using Reciprocal Rank Fusion (RRF)
combined_score = []

for i in range(0, len(ranking_semantic_search), 1):
    k = 60
    rrf_score = 1 / (k + ranking_keyword_search[i]) + 1 / (
        k + ranking_semantic_search[i]
    )
    combined_score.append(rrf_score)

combined_score_df = pd.DataFrame(combined_score, columns=["combined_score"])
new_ranking = (
    combined_score_df.sort_values(by=["combined_score"], ascending=False).index + 1
)

### How to optimize similarity search by combining similarity search and keyword search

In [10]:
from rank_bm25 import BM25Okapi
import pandas as pd

text_chunks = [
    "The Great Fire of London in 1666 destroyed over 13,000 houses.",
    "Julius Caesar was assassinated on the Ides of March (March 15) in 44 BCE.",
    "The Black Death is estimated to have killed nearly one-third of the European population.",
]

# Tokenize text into words
tokenized_chunks = [chunk.split(" ") for chunk in text_chunks]

bm25 = BM25Okapi(tokenized_chunks)

user_query = "Tell me something interesting about diseases in history"
tokenized_query = user_query.split(" ")

# BM25 scores for each document
bm25_scores = bm25.get_scores(tokenized_query)

# Document IDs ordered by keyword relevance (best first)
keyword_ranking_doc_ids = (
    pd.DataFrame(bm25_scores, columns=["score"])
    .sort_values(by="score", ascending=False)
    .index
    .to_list()
)

In [11]:
from sentence_transformers.util import cos_sim
from openai import OpenAI

client = OpenAI()

def create_embedding(text):
    return (
        client.embeddings.create(
            input=[text], model="text-embedding-3-small"
        )
        .data[0]
        .embedding
    )

embeddings_df = pd.DataFrame(text_chunks, columns=["text_chunk"])
embeddings_df["embedding"] = embeddings_df["text_chunk"].apply(create_embedding)

query_embedding = create_embedding(user_query)

# Compute cosine similarity
embeddings_df["similarity"] = cos_sim(
    embeddings_df["embedding"], query_embedding
)

# Document IDs ordered by semantic similarity (best first)
semantic_ranking_doc_ids = (
    embeddings_df
    .sort_values(by="similarity", ascending=False)
    .index
    .to_list()
)

In [ ]:
# ranking_semantic_search, ranking_keyword_search = hybrid_search()

In [ ]:
# ranking_semantic_search, ranking_keyword_search

In [12]:
# Reciprocal Rank Fusion parameter
# Higher k reduces the influence of very top ranks
k = 60

# Build a table with one row per document
rrf_df = pd.DataFrame({"doc_id": range(len(text_chunks))})

# Map document IDs to their rank positions
keyword_rank_map = {
    doc_id: rank for rank, doc_id in enumerate(keyword_ranking_doc_ids, start=1)
}
semantic_rank_map = {
    doc_id: rank for rank, doc_id in enumerate(semantic_ranking_doc_ids, start=1)
}

rrf_df["keyword_rank"] = rrf_df["doc_id"].map(keyword_rank_map)
rrf_df["semantic_rank"] = rrf_df["doc_id"].map(semantic_rank_map)

# Reciprocal Rank Fusion score
rrf_df["rrf_score"] = (
    1 / (k + rrf_df["keyword_rank"]) +
    1 / (k + rrf_df["semantic_rank"])
)

# Final hybrid ranking
rrf_df = rrf_df.sort_values("rrf_score", ascending=False)

final_ranking_doc_ids = rrf_df["doc_id"].to_list()

In [13]:
new_ranking

Index([2, 1, 3], dtype='int64')

### Performing Text Classification Using Embeddings

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import PyPDF2
import pandas as pd

pdf_files = [
    {
        "file_path": "../datasets/pdf_files/history_of_deep_learning.pdf",
        "label": "Deep_Learning",
    },
    {
        "file_path": "../datasets/pdf_files/premier_league_history.pdf",
        "label": "Premier_League",
    },
]

chunks_dict_list = []

# split both documents into chunks and append to the same list of dicts
for pdf_file in pdf_files:
    with open(pdf_file["file_path"], "rb") as file:
        reader = PyPDF2.PdfReader(file)

        text = ""
        for page in reader.pages:
            text += page.extract_text()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=0,
        length_function=len,
        is_separator_regex=False,
    )

    chunks = text_splitter.split_text(text)

    for chunk in chunks:
        chunks_dict_list.append({"text": chunk, "label": pdf_file["label"]})

chunks_df = pd.DataFrame(chunks_dict_list)

In [17]:
client = OpenAI()

def create_embeddings(text_chunk, client):
    response = client.embeddings.create(
        input=[text_chunk], model="text-embedding-3-small"
    )
    embedding = response.data[0].embedding
    return embedding

chunks_df["embedding"] = chunks_df["text"].apply(create_embeddings, client=client)

In [18]:
chunks_df.to_csv("chunks_df.csv")

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

y = chunks_df["label"]
X = chunks_df["embedding"].apply(
    lambda x: pd.Series(eval(x)) if isinstance(x, str) else pd.Series(x)
)

# Train a random forest classifier
clf = RandomForestClassifier()
clf.fit(X, y)

RandomForestClassifier()

In [20]:
test_embedding = create_embeddings(
    text_chunk="What is the name of the top football league in England?",
    client=client,
)

X_test = [test_embedding]

# Predict the most likely class
predicted_classes = clf.predict(X_test)  # e.g. ['Premier_League']
probabilities = clf.predict_proba(X_test)  # e.g. array([[0.06, 0.94]])